TASK 2: TRANSFORM (transform.py)

Nhiệm vụ: Đọc bronze_rows từ task ingest, làm sạch tạo Silver, tổng hợp tạo Gold, ghi tiếp silver_rows và gold_rows vào taskValues.

In [0]:
TEN = "duykh"
BRONZE = f"workspace.bronze_{TEN}.raw_orders_{TEN}"
SILVER = f"workspace.silver_{TEN}.fnd_orders_{TEN}"
GOLD   = f"workspace.gold_{TEN}.fact_daily_shop_sales_{TEN}"

# 1. Đọc taskValues từ task "ingest" (debugValue dùng khi chạy thủ công notebook để test)
bronze_rows = dbutils.jobs.taskValues.get(taskKey="ingest", key="bronze_rows", debugValue=35)
print(f"Nhận từ task ingest: bronze_rows = {bronze_rows}")

# 2. Xử lý tầng SILVER: Ép kiểu an toàn cho amount bằng TRY_CAST(... AS DOUBLE)
spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER} AS
WITH dedup AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) AS rn
  FROM {BRONZE}
  WHERE order_id IS NOT NULL AND trim(order_id) <> ''
),
cleaned AS (
  SELECT
    order_id,
    trim(shop_code)                    AS shop_code,
    order_ts,
    TRY_CAST(trim(amount) AS DOUBLE)   AS amount,
    TRY_CAST(trim(quantity) AS INT)    AS quantity,
    product_id,
    upper(trim(status))                AS status_norm,
    channel,
    updated_at,
    _ingested_at,
    _source_file
  FROM dedup
  WHERE rn = 1
)
SELECT
  *,
  CASE WHEN amount IS NULL THEN 'AMOUNT_KHONG_HOP_LE' END AS data_issue
FROM cleaned;
""")

# 3. Xử lý tầng GOLD: Tổng hợp theo shop x ngày, gom null thành UNKNOWN
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD} AS
SELECT
  COALESCE(shop_code, 'UNKNOWN') AS shop_code,
  CAST(order_ts AS DATE)         AS order_date,
  COUNT(*)                       AS so_don,
  SUM(amount)                    AS revenue
FROM {SILVER}
WHERE data_issue IS NULL
  AND status_norm = 'PAID'
  AND amount > 0
GROUP BY COALESCE(shop_code, 'UNKNOWN'), CAST(order_ts AS DATE);
""")

# 4. Đếm số dòng Silver và Gold để gửi sang task validate
silver_count = spark.table(SILVER).count()
gold_count   = spark.table(GOLD).count()
print(f"Số dòng Silver: {silver_count} | Số dòng Gold: {gold_count}")

dbutils.jobs.taskValues.set(key="silver_rows", value=silver_count)
dbutils.jobs.taskValues.set(key="gold_rows", value=gold_count)
print("Đã ghi taskValues: silver_rows và gold_rows!")